

### **Cell 1: Setup and Libraries**

This cell imports the necessary libraries and checks the TensorFlow version.

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
from tensorflow.keras.preprocessing import image

print(tf.__version__)

2.19.0


In [ ]:
import zipfile
import os

# Define the path to your zipped file
zip_file_path = '/content/Cat_Dog_train_test_dataset-20260217T052024Z-1-001.zip' # This path should point to your actual zip file

# Define the directory where you want to extract the contents
extraction_path = './' # Extracts to the current working directory

# Create the extraction directory if it doesn't exist
if not os.path.exists(extraction_path):
    os.makedirs(extraction_path)

# Unzip the file
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extraction_path)

print(f'Successfully unzipped {zip_file_path} to {extraction_path}')

Successfully unzipped /content/Cat_Dog_train_test_dataset-20260217T052024Z-1-001.zip to ./


### **Cell 2: Data Preprocessing**



In [ ]:
# Part 1 - Data Preprocessing

# Preprocessing the Training set
# We apply transformations (shear, zoom, flip) to avoid overfitting
train_datagen = ImageDataGenerator(
    rescale = 1./255,
    shear_range = 0.2,
    zoom_range = 0.2,
    horizontal_flip = True
)

# Load training data
# Ensure 'dataset/training_set' exists in your Colab files
training_set = train_datagen.flow_from_directory(
    '/content/Cat_Dog_train_test_dataset/train',
    target_size = (64, 64),
    batch_size = 32,
    class_mode = 'binary'
)

# Preprocessing the Test set
# We only rescale the test pixels (no flipping or zooming)
test_datagen = ImageDataGenerator(rescale = 1./255)

# Load test data
test_set = test_datagen.flow_from_directory(
    '/content/Cat_Dog_train_test_dataset/test',
    target_size = (64, 64),
    batch_size = 32,
    class_mode = 'binary'
)

Found 800 images belonging to 2 classes.
Found 200 images belonging to 2 classes.


### **Cell 3: Building the CNN**

This cell defines the architecture of your Neural Network.

In [ ]:
# Part 2 - Building the CNN

# Initialising the CNN
cnn = tf.keras.models.Sequential()

# Step 1 - Convolution
# Note: input_shape matches the target_size defined above (64, 64) + 3 for color channels (RGB)
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu', input_shape=[64, 64, 3]))

# Step 2 - Pooling
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

# Adding a second convolutional layer
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu'))
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

# Step 3 - Flattening
cnn.add(tf.keras.layers.Flatten())

# Step 4 - Full Connection
cnn.add(tf.keras.layers.Dense(units=128, activation='relu'))

# Step 5 - Output Layer (1 unit for binary classification)
cnn.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


### **Cell 4: Training the Model**

This cell compiles and starts the training process.

In [ ]:
# Part 3 - Training the CNN

# Compiling the CNN
cnn.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

# Training the CNN on the Training set and evaluating it on the Test set
cnn.fit(x = training_set, validation_data = test_set, epochs = 25)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 7s 134ms/step - accuracy: 0.4617 - loss: 0.7385 - val_accuracy: 0.5000 - val_loss: 0.6921
Epoch 2/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 87ms/step - accuracy: 0.5138 - loss: 0.6919 - val_accuracy: 0.5800 - val_loss: 0.6911
Epoch 3/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 121ms/step - accuracy: 0.5881 - loss: 0.6898 - val_accuracy: 0.6200 - val_loss: 0.6834
Epoch 4/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 88ms/step - accuracy: 0.6206 - loss: 0.6797 - val_accuracy: 0.5250 - val_loss: 0.6790
Epoch 5/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 88ms/step - accuracy: 0.5507 - loss: 0.6854 - val_accuracy: 0.6000 - val_loss: 0.6578
Epoch 6/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 87ms/step - accuracy: 0.6626 - loss: 0.6447 - val_accuracy: 0.6400 - val_loss: 0.6521
Epoch 7/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 87ms/step - accuracy: 0.6719 - loss: 0.6203 - val_accuracy: 0.6950 - val_loss: 0.6281
Epoch 8/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 121ms/step - accuracy: 0.6949 - loss: 0.5868 - val_accuracy: 0.5350 

### **Cell 5: Single Prediction**

Use this cell to upload a specific image (e.g., `cat_or_dog_1.jpg`) and check if it is a cat or a dog.

In [ ]:
# Part 4 - Making a single prediction

# Load the image to test
# Ensure 'dataset/single_prediction/cat_or_dog_1.jpg' exists
test_image = image.load_img('/content/Cat_Dog_train_test_dataset/test/dogs/dog.4156.jpg', target_size = (64, 64))

# Convert image to array and expand dimensions to match batch format
test_image = image.img_to_array(test_image)
test_image = np.expand_dims(test_image, axis = 0)

# Make prediction
result = cnn.predict(test_image)

# Check indices to see which number corresponds to which animal
# print(training_set.class_indices)

# Logic to print the result
if result[0][0] == 1:
    prediction = 'cats'
else:
    prediction = 'dogs'

print(prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
cats
